# Phase 2 — LM Judge: Classify Multi-Turn MedQA Clarifying Questions

Classifies all 300 clarifying questions (100 cases × 3 CQs) as **EPISTEMIC** or **ALEATORIC**.

Reads:  `outputs/medqa/gemini-2.5-flash/phase1_multiturn_results.csv`  
Writes: `outputs/medqa/gemini-2.5-flash/phase1_multiturn_classified.csv` (300 rows, long format)

Each row in the output has: `id`, `turn` (1/2/3), `clarifying_question`, `label`, `raw_response`

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

DATASET              = "medqa"
ROOT                 = Path("../../").resolve()
PROMPTS_DIR          = ROOT / "prompts" / DATASET

MODEL_ID             = "gemini-2.5-flash"
OUTPUTS_DIR          = ROOT / "outputs" / DATASET / MODEL_ID

JUDGE_INSTRUCTION    = PROMPTS_DIR / "judge_instruction.txt"
PHASE1_RESULTS_PATH  = OUTPUTS_DIR / "phase1_multiturn_results.csv"
OUTPUT_PATH          = OUTPUTS_DIR / "phase1_multiturn_classified.csv"
INPUT_CSV            = OUTPUTS_DIR / "phase1_multiturn_cq_input.csv"

REQUEST_INTERVAL     = 1.0
REGENERATE           = False

import logging
import pandas as pd
from collections import Counter
from IPython.display import display

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.judge import LLMJudge, CSVBatchClassifier, FewShotExample

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

Environment loaded.


## Few-Shot Examples (same as single-turn judge — no leakage risk, hand-crafted)

In [2]:
FEW_SHOT_EXAMPLES: list[FewShotExample] = [
    FewShotExample(
        input="You mentioned having 'MCAS' — could you describe what symptoms it causes for you or what your doctor told you about it?",
        expected_output="EPISTEMIC",
        explanation="The model doesn't have enough context about this entity to reason clinically. There is a definite answer — once the patient explains it, the knowledge gap is fully and permanently resolved.",
    ),
    FewShotExample(
        input="You described the rash as both 'spreading' and 'fading' — is it currently getting larger or is it clearing up?",
        expected_output="EPISTEMIC",
        explanation="The two descriptions contradict each other, making clinical categorisation impossible. There is one correct factual state right now — the model is resolving a factual contradiction, not a preference.",
    ),
    FewShotExample(
        input="When you say you feel 'weak', do you mean you have no energy and feel fatigued, or that you have actual muscle weakness and difficulty lifting things?",
        expected_output="ALEATORIC",
        explanation="'Weak' has two clinically distinct meanings (fatigue vs true motor weakness) that point to completely different differentials. No external fact can resolve which meaning the patient intends — only the patient can.",
    ),
    FewShotExample(
        input="When you said 'it started after the procedure', are you referring to the chest pain or the shortness of breath?",
        expected_output="ALEATORIC",
        explanation="The pronoun 'it' could validly refer to either symptom. No external fact can determine which one the patient meant — only the patient's own context resolves this.",
    ),
    FewShotExample(
        input="Which aspect of your recovery matters most to you — getting back to work quickly, minimising pain, or avoiding surgery?",
        expected_output="ALEATORIC",
        explanation="The answer depends entirely on this individual patient's values and priorities. No clinical fact or external knowledge can determine their personal preference — it is irreducibly patient-specific.",
    ),
    FewShotExample(
        input="When you ask about treatment options, are you looking for information about medications, surgical approaches, or lifestyle changes?",
        expected_output="ALEATORIC",
        explanation="The request is underspecified — multiple valid interpretations exist and the correct path depends entirely on what the patient wants, not on any clinical fact.",
    ),
    FewShotExample(
        input="When you say your symptoms are 'intermittent', do you mean they come and go throughout the day, or that you have symptom-free periods lasting weeks?",
        expected_output="ALEATORIC",
        explanation="Two valid temporal interpretations exist, each with different clinical significance. Only the patient knows which pattern applies — no external fact resolves this.",
    ),
    FewShotExample(
        input="When you say the pain is 'everywhere', do you mean it is diffuse throughout your abdomen, or that it shifts between different locations?",
        expected_output="ALEATORIC",
        explanation="Two spatially distinct clinical patterns (diffuse vs migratory pain) are both plausible readings. Only the patient can clarify which pattern they actually experience.",
    ),
]

print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex.expected_output}] {ex.input[:80]}")

Few-shot examples: 8
  [EPISTEMIC] You mentioned having 'MCAS' — could you describe what symptoms it causes for you
  [EPISTEMIC] You described the rash as both 'spreading' and 'fading' — is it currently gettin
  [ALEATORIC] When you say you feel 'weak', do you mean you have no energy and feel fatigued, 
  [ALEATORIC] When you said 'it started after the procedure', are you referring to the chest p
  [ALEATORIC] Which aspect of your recovery matters most to you — getting back to work quickly
  [ALEATORIC] When you ask about treatment options, are you looking for information about medi
  [ALEATORIC] When you say your symptoms are 'intermittent', do you mean they come and go thro
  [ALEATORIC] When you say the pain is 'everywhere', do you mean it is diffuse throughout your


## Smoke Test

In [3]:
provider = GeminiProvider(model_id=MODEL_ID)

judge = LLMJudge(
    provider=provider,
    instructions_path=JUDGE_INSTRUCTION,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    label_parser=lambda text: text.strip().upper(),
)

smoke = judge.evaluate("Have you been diagnosed with hypertension or any other heart condition before?")
print(f"Smoke test → label={smoke.label}, error={smoke.error}")
assert smoke.label == "EPISTEMIC", f"Unexpected smoke test label: {smoke.label}"
print("Smoke test passed.")

17:35:02 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


17:35:02 [INFO] src.judge — LLMJudge ready — provider=gemini model=gemini-2.5-flash few_shot_count=8


17:35:02 [INFO] src.judge — Evaluating: 'Have you been diagnosed with hypertension or any other heart...'


17:35:02 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:35:05 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:35:05 [INFO] src.judge — label='EPISTEMIC' latency=3.294s


Smoke test → label=EPISTEMIC, error=None
Smoke test passed.


## Build 300-Row Long-Format Input

One row per CQ (turn 1, 2, 3) per case — 100 cases × 3 = 300 rows.

In [4]:
phase1_df = pd.read_csv(PHASE1_RESULTS_PATH)
valid_df  = phase1_df[~phase1_df["was_blocked"]].copy()

print(f"Phase 1 rows: {len(phase1_df)} | Blocked: {phase1_df['was_blocked'].sum()}")
print(f"Usable: {len(valid_df)}")

# Melt the three CQ columns into long format
rows = []
for _, r in valid_df.iterrows():
    for turn in range(1, 4):
        cq_col = f"cq_{turn}"
        cq = r[cq_col]
        if pd.notna(cq) and str(cq).strip():
            rows.append({"id": r["id"], "turn": turn, "clarifying_question": cq})

long_df = pd.DataFrame(rows)
long_df.to_csv(INPUT_CSV, index=False)
print(f"Long-format input: {len(long_df)} rows saved to {INPUT_CSV.name}")
display(long_df.head(6))

# Scientific validity: each case should have exactly 3 CQs
cqs_per_case = long_df.groupby("id").size()
assert (cqs_per_case == 3).all(), f"Some cases have != 3 CQs: {cqs_per_case[cqs_per_case != 3]}"
print("Validity check PASSED — every case has exactly 3 CQs")

Phase 1 rows: 100 | Blocked: 0
Usable: 100
Long-format input: 300 rows saved to phase1_multiturn_cq_input.csv


,id,turn,clarifying_question
0,medqa_0982,1,What is the assumed diagnosis in this case?
1,medqa_0982,2,Are there any signs or symptoms of systemic in...
2,medqa_0982,3,Are there any specific imaging findings beyond...
3,medqa_0799,1,What are the patient's current blood pressure ...
4,medqa_0799,2,What is the patient's cardiac rhythm?
5,medqa_0799,3,What are the EKG findings?


Validity check PASSED — every case has exactly 3 CQs


## Classify All 300 Clarifying Questions

In [5]:
if REGENERATE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Deleted existing output — regenerating.")

classifier = CSVBatchClassifier(
    judge=judge,
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_PATH,
    question_column="clarifying_question",
    id_column="id",
    delay_between_calls=REQUEST_INTERVAL,
)
classifier.run()
print(f"Classification complete → {OUTPUT_PATH}")

17:35:05 [INFO] src.judge — Evaluating: 'What is the assumed diagnosis in this case?...'


17:35:05 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:35:07 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:35:07 [INFO] src.judge — label='EPISTEMIC' latency=2.090s


17:35:08 [INFO] src.judge — Evaluating: 'Are there any signs or symptoms of systemic inflammation, su...'


...

17:49:47 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:49:48 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:49:48 [INFO] src.judge — label='EPISTEMIC' latency=1.708s


17:49:49 [INFO] src.judge — Batch complete — total=300 classified=300 skipped=0 errors=0


Classification complete → D:\final_project\pilot_study\outputs\medqa\gemini-2.5-flash\phase1_multiturn_classified.csv


## Results Summary

In [6]:
classified_df = pd.read_csv(OUTPUT_PATH)
# Re-attach turn number by matching on id + question text (classifier renames cq column to "question")
turn_lookup = long_df.rename(columns={"clarifying_question": "question"})
classified_df = classified_df.merge(turn_lookup[["id", "turn", "question"]], on=["id", "question"], how="left")

valid_labels = {"EPISTEMIC", "ALEATORIC"}
classified = classified_df[classified_df["label"].isin(valid_labels)]

print(f"Total classified: {len(classified_df)} | Errors: {(classified_df['error'].notna() & (classified_df['error'] != '')).sum()}")
print(f"Valid labels: {len(classified)}")
print()

print("Overall label distribution:")
print(classified["label"].value_counts().to_string())
print()

print("Label distribution by turn:")
display(
    classified.groupby(["turn", "label"]).size().unstack(fill_value=0)
)
print()

phase1_for_merge = phase1_df.copy()
print("Correctness by label and turn:")
for turn in range(1, 4):
    turn_df = classified[classified["turn"] == turn].copy()
    correct_col = f"is_correct_{turn}" if turn < 3 else "is_correct_final"
    merged_turn = turn_df.merge(
        phase1_for_merge[["id", correct_col, "difficulty"]],
        on="id", how="left"
    )
    print(f"  Turn {turn} correctness by label:")
    display(
        merged_turn.groupby("label")[correct_col].agg(["mean", "count"]).round(3)
    )

print()
print("Sample classifications (first 9):")
display(classified[["id", "turn", "label", "question"]].head(9))


Total classified: 300 | Errors: 0
Valid labels: 300

Overall label distribution:
label
EPISTEMIC    293
ALEATORIC      7

Label distribution by turn:


label,ALEATORIC,EPISTEMIC
turn,,
1,4,96
2,2,98
3,1,99



Correctness by label and turn:
  Turn 1 correctness by label:


,mean,count
label,,
ALEATORIC,0.750,4
EPISTEMIC,0.667,96


  Turn 2 correctness by label:


,mean,count
label,,
ALEATORIC,1.000,2
EPISTEMIC,0.796,98


  Turn 3 correctness by label:


,mean,count
label,,
ALEATORIC,1.000,1
EPISTEMIC,0.828,99



Sample classifications (first 9):


,id,turn,label,question
0,medqa_0982,1,EPISTEMIC,What is the assumed diagnosis in this case?
1,medqa_0982,2,EPISTEMIC,Are there any signs or symptoms of systemic in...
2,medqa_0982,3,EPISTEMIC,Are there any specific imaging findings beyond...
3,medqa_0799,1,EPISTEMIC,What are the patient's current blood pressure ...
4,medqa_0799,2,EPISTEMIC,What is the patient's cardiac rhythm?
5,medqa_0799,3,EPISTEMIC,What are the EKG findings?
6,medqa_1095,1,EPISTEMIC,What specific pathology was identified on the ...
7,medqa_1095,2,EPISTEMIC,Is the defect located on the medial or lateral...
8,medqa_1095,3,EPISTEMIC,Does the CT scan report indicate any involveme...
